# Notebook 2/3 — Building the relationships (prerequisite for the point-in-time logic)

Starting from the `Dossier`/identifiers graph, we build the structure that lets us retrieve a community **at any date, without leaking the future**:

1. **`SIMILARITE`** (`Dossier → Dossier`) — two dossiers sharing an identifier are linked. This is the base similarity graph (client equivalent).
2. **`COMPONENT_PARENT`** — a time-oriented union-find forest: each merge points forward.
3. **`DFS_NEXT`** — a pre-materialized linked list of a community's members (constant-time traversal).
4. **`LAST_DFS_NODE_IN_COMP`** — end-of-chain marker (bounded traversal).

> **Eventually, this notebook goes away.** In production, these relationships will be maintained **on the fly** by an atomic Cypher query for each new dossier (*online* ingestion, see README §"Online"). Here we build them in *batch* over the existing history.

Prerequisite: having run **Notebook 1** (or having the real client graph). The **GDS** library must be installed on the Neo4j instance.


In [6]:
# Cell 1: Connection

!pip install neo4j -q

import time
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "fraudwcctemporal"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Connected to Neo4j")

# List of bipartite Dossier -> identifier relationships used for similarity
BIPARTITE = "EMAIL_ROOT|TELEPHONE|ADRESSE_IP|DEVICE|CARTE_BANCAIRE|COMPTE|ETAT_CIVIL|ADRESSE_EMPRUNTEUR|FOYER_SICLID"


✅ Connected to Neo4j


In [7]:
# Cell 2: Build SIMILARITE (Dossier -> Dossier) from shared identifiers
#
# Method (batched by WCC for performance):
#   - project the bipartite Dossier/identifiers graph
#   - WCC to group by component
#   - for each identifier, link its CONSECUTIVE dossiers in time (oldest -> most recent)

print("Building SIMILARITE...")

with driver.session(database=NEO4J_DATABASE) as session:
    print("  Step 1: project the bipartite graph")
    start = time.time()
    session.run("""
        CYPHER runtime=parallel
        MATCH (source:Dossier)
        OPTIONAL MATCH (source)-[:BIPARTITE_RELS]->(target)
        RETURN gds.graph.project('dossier_ident_graph', source, target, {})
    """.replace("BIPARTITE_RELS", BIPARTITE))
    print(f"     ✓ ({time.time()-start:.2f}s)")

    print("  Step 2: SIMILARITE via consecutive pairs per identifier")
    start = time.time()
    session.run("""
        CALL gds.wcc.stream('dossier_ident_graph')
        YIELD nodeId, componentId
        WITH gds.util.asNode(nodeId) AS n, componentId AS community
        WHERE NOT n:Dossier
        WITH community, collect(n) AS idents
        CALL (idents) {
          UNWIND idents AS ident
          CALL (ident) {
            MATCH (d:Dossier)-[:BIPARTITE_RELS]->(ident)
            WITH DISTINCT d ORDER BY d.DATE_COMMANDE
            WITH collect(d) AS ds
            UNWIND range(0, size(ds)-2) AS ix
            WITH ds[ix] AS source, ds[ix+1] AS target
            MERGE (source)-[:SIMILARITE]->(target)
          }
        } IN 8 CONCURRENT TRANSACTIONS OF 100 ROWS
    """.replace("BIPARTITE_RELS", BIPARTITE))
    print(f"     ✓ ({time.time()-start:.2f}s)")

    session.run("CALL gds.graph.drop('dossier_ident_graph', false) YIELD graphName RETURN graphName")
    n = session.run("MATCH ()-[r:SIMILARITE]->() RETURN count(r) AS n").single()["n"]
    print(f"\n✅ {n:,} SIMILARITE relationships created")


Building SIMILARITE...
  Step 1: project the bipartite graph
     ✓ (0.05s)
  Step 2: SIMILARITE via consecutive pairs per identifier


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('dossier_ident_graph', false)"


     ✓ (34.59s)

✅ 231,959 SIMILARITE relationships created


In [8]:
# Cell 3: Build COMPONENT_PARENT (time-oriented union-find forest)
#
# We replay the online algorithm in batch: within each WCC component, dossiers are
# processed by increasing DATE_COMMANDE; each attaches to the current head of its community.

print("Building COMPONENT_PARENT...")

with driver.session(database=NEO4J_DATABASE) as session:
    print("  Step 1: project the SIMILARITE graph")
    start = time.time()
    session.run("""
        CYPHER runtime=parallel
        MATCH (source:Dossier)
        OPTIONAL MATCH (source)-[:SIMILARITE]->(target)
        RETURN gds.graph.project('similarite_graph', source, target, {})
    """)
    print(f"     ✓ ({time.time()-start:.2f}s)")

    print("  Step 2: COMPONENT_PARENT forest")
    start = time.time()
    session.run("""
        CALL gds.wcc.stream('similarite_graph')
        YIELD nodeId, componentId
        WITH gds.util.asNode(nodeId) AS event, componentId
        WITH componentId, collect(event) AS events
        ORDER BY rand()
        CALL (events) {
          UNWIND events AS e
          WITH e WHERE NOT e:ComponentNode
          ORDER BY e.DATE_COMMANDE ASC
          CALL (e) {
            SET e:ComponentNode
            WITH e
            MATCH (x:ComponentNode)-[:SIMILARITE]->(e)
            MATCH (x)-[:COMPONENT_PARENT]->*(cc WHERE NOT EXISTS {(cc)-[:COMPONENT_PARENT]->()})
            MERGE (cc)-[:COMPONENT_PARENT]->(e)
          }
        } IN 8 CONCURRENT TRANSACTIONS OF 100 ROWS
    """)
    print(f"     ✓ ({time.time()-start:.2f}s)")

    session.run("CALL gds.graph.drop('similarite_graph', false) YIELD graphName RETURN graphName")
    n = session.run("MATCH ()-[r:COMPONENT_PARENT]->() RETURN count(r) AS n").single()["n"]
    print(f"\n✅ {n:,} COMPONENT_PARENT relationships created")


Building COMPONENT_PARENT...
  Step 1: project the SIMILARITE graph
     ✓ (0.02s)
  Step 2: COMPONENT_PARENT forest


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('similarite_graph', false)"


     ✓ (12.82s)

✅ 231,959 COMPONENT_PARENT relationships created


In [9]:
# Cell 4: Build DFS_NEXT + LAST_DFS_NODE_IN_COMP (constant-time community traversal)

print("Building DFS_NEXT + LAST_DFS_NODE_IN_COMP...")

with driver.session(database=NEO4J_DATABASE) as session:
    print("  Step 1: project the COMPONENT_PARENT forest (reversed for DFS from the roots)")
    start = time.time()
    session.run("""
        MATCH (source:Dossier)
        OPTIONAL MATCH (source)<-[:COMPONENT_PARENT]-(target)
        RETURN gds.graph.project('component_forest', source, target, {})
    """)
    print(f"     ✓ ({time.time()-start:.2f}s)")

    print("  Step 2: DFS_NEXT chains from each root")
    start = time.time()
    session.run("""
        MATCH (source:ComponentNode)
        WHERE NOT EXISTS {(source)-[:COMPONENT_PARENT]->()}
          AND EXISTS {(source)<-[:COMPONENT_PARENT]-()}
        CALL (source) {
          CALL gds.dfs.stream('component_forest', { sourceNode: source })
          YIELD path
          WITH relationships(path) AS rels
          UNWIND rels AS rel
          WITH startNode(rel) AS n1, endNode(rel) AS n2
          MERGE (n1)-[:DFS_NEXT]->(n2)
        } IN 8 CONCURRENT TRANSACTIONS OF 50 ROWS
    """)
    print(f"     ✓ ({time.time()-start:.2f}s)")

    print("  Step 3: LAST_DFS_NODE_IN_COMP markers")
    start = time.time()
    session.run("""
    CYPHER 25
    CALL () {
        MATCH (c1:ComponentNode)-[:DFS_NEXT]->(c2:ComponentNode)
        MATCH (c1)(()-[:COMPONENT_PARENT]->(ps)
          WHERE NOT EXISTS {(c2)-[:COMPONENT_PARENT]->*(ps)}
        )*
        UNWIND ps AS p
        RETURN c1, p
      UNION
        MATCH (c1:ComponentNode WHERE NOT EXISTS {(c1)-[:DFS_NEXT]->()})
        MATCH (c1)(()-[:COMPONENT_PARENT]->(ps))*
        UNWIND ps AS p
        RETURN c1, p
      UNION
        MATCH (c1:ComponentNode WHERE NOT EXISTS {()-[:COMPONENT_PARENT]->(c1)})
        RETURN c1, c1 AS p
    }
    CALL (c1, p) {
      MERGE (p)-[:LAST_DFS_NODE_IN_COMP]->(c1)
    } IN TRANSACTIONS OF 100 ROWS
    """)
    print(f"     ✓ ({time.time()-start:.2f}s)")

    session.run("CALL gds.graph.drop('component_forest', false) YIELD graphName RETURN graphName")
    r1 = session.run("MATCH ()-[r:DFS_NEXT]->() RETURN count(r) AS n").single()["n"]
    r2 = session.run("MATCH ()-[r:LAST_DFS_NODE_IN_COMP]->() RETURN count(r) AS n").single()["n"]
    print(f"\n✅ {r1:,} DFS_NEXT  |  {r2:,} LAST_DFS_NODE_IN_COMP")


Building DFS_NEXT + LAST_DFS_NODE_IN_COMP...
  Step 1: project the COMPONENT_PARENT forest (reversed for DFS from the roots)
     ✓ (0.03s)
  Step 2: DFS_NEXT chains from each root
     ✓ (11.34s)
  Step 3: LAST_DFS_NODE_IN_COMP markers


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('component_forest', false)"


     ✓ (59.70s)

✅ 231,959 DFS_NEXT  |  1,000,000 LAST_DFS_NODE_IN_COMP


In [10]:
# Cell 5: Sanity check — community sizes + point-in-time snapshot
#
# ⚠️ A community's size = LENGTH of the DFS_NEXT chain (linked list),
#    NOT the out-degree (which is always 0 or 1). We start from the current heads
#    (dossiers with no outgoing COMPONENT_PARENT) and count the whole chain.

with driver.session(database=NEO4J_DATABASE) as session:
    tops = session.run("""
        MATCH (head:Dossier)
        WHERE NOT EXISTS {(head)-[:COMPONENT_PARENT]->()}
          AND EXISTS {(head)-[:DFS_NEXT]->()}
        CALL (head) {
          MATCH (head)-[:DFS_NEXT]->*(x)
          RETURN count(x) AS taille
        }
        RETURN head.NODOS AS nodos, taille
        ORDER BY taille DESC LIMIT 10
    """).data()

print("Top 10 largest communities:")
for t in tops:
    print(f"   {t['nodos']}  ->  {t['taille']} dossiers")

if tops:
    nodos = tops[0]["nodos"]
    with driver.session(database=NEO4J_DATABASE) as session:
        comm = session.run("""
            MATCH (d:Dossier {NODOS: $nodos})(()-[:DFS_NEXT]->(m))*(last)<-[:LAST_DFS_NODE_IN_COMP]-(d)
            UNWIND [d] + m AS ev
            RETURN ev.NODOS AS nodos, toString(ev.DATE_COMMANDE) AS date,
                   ev.TOP_FRAUDE AS fraude, ev.pattern AS pattern
            ORDER BY date
        """, nodos=nodos).data()
    print(f"\nPoint-in-time snapshot of {nodos}: {len(comm)} dossiers")
    for c in comm[:15]:
        print(f"   {c['nodos']}  {c['date'][:10]}  fraude={c['fraude']}  {c['pattern']}")
else:
    print("No community of size > 1: did you run Notebook 1 and then cells 2-4 above?")

print("\n👉 Go to Notebook 3 for the walk-forward training.")


Top 10 largest communities:
   DOS_00875908  ->  649 dossiers
   DOS_00554920  ->  639 dossiers
   DOS_00177946  ->  605 dossiers
   DOS_00442063  ->  596 dossiers
   DOS_00732701  ->  565 dossiers
   DOS_00451326  ->  557 dossiers
   DOS_00552560  ->  540 dossiers
   DOS_00252982  ->  509 dossiers
   DOS_00232150  ->  484 dossiers
   DOS_00674760  ->  440 dossiers

Point-in-time snapshot of DOS_00875908: 649 dossiers
   DOS_00876105  2024-07-31  fraude=True  fraude_grande
   DOS_00875843  2024-08-01  fraude=False  fraude_grande
   DOS_00875962  2024-08-02  fraude=False  fraude_grande
   DOS_00875894  2024-08-03  fraude=True  fraude_grande
   DOS_00876171  2024-08-03  fraude=True  fraude_grande
   DOS_00876349  2024-08-05  fraude=True  fraude_grande
   DOS_00876262  2024-08-05  fraude=True  fraude_grande
   DOS_00876325  2024-08-06  fraude=True  fraude_grande
   DOS_00875918  2024-08-06  fraude=True  fraude_grande
   DOS_00876382  2024-08-07  fraude=False  fraude_grande
   DOS_00876080